# Phase 1: Advanced EDA (v2)
### Understanding Dataset Geometry, Statistics, and Quality

This notebook provides a deep dive into the 50k unified dataset. Unlike basic EDA, we focus here on **Bounding Box (BBox) statistics** and **Spatial Distributions** which directly impact model training performance.

### 1. Environment Setup

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import fiftyone as fo
from pathlib import Path
from tqdm import tqdm

# --- Project Root Discovery ---
def get_project_root():
    current_path = Path().resolve()
    for parent in [current_path] + list(current_path.parents):
        if (parent / "requirements.txt").exists():
            return parent
    return current_path

PROJECT_ROOT = get_project_root()
DATASET_DIR = PROJECT_ROOT / "data" / "processed" / "yolo_dataset"
print(f"✅ Root identified: {PROJECT_ROOT}")

### 2. Loading Unified Dataset
We load the 'test' split for a representative statistical sample.

In [ ]:
dataset_name = "Weapon_Detection_Advanced_EDA"
if dataset_name in fo.list_datasets():
    fo.delete_dataset(dataset_name)

dataset = fo.Dataset.from_dir(
    dataset_dir=str(DATASET_DIR),
    dataset_type=fo.types.YOLOv5Dataset,
    name=dataset_name,
    split="test"
)

print(f"Loaded {len(dataset)} images from the test split.")

### 3. Bounding Box Geometry Analysis
We extract every bounding box to analyze their **Relative Area** and **Aspect Ratio**.

In [ ]:
bbox_data = []
for sample in tqdm(dataset, desc="Extracting BBoxes"):
    if sample.ground_truth:
        for det in sample.ground_truth.detections:
            # YOLO format in FiftyOne is [x_top_left, y_top_left, width, height] (normalized 0-1)
            w = det.bounding_box[2]
            h = det.bounding_box[3]
            bbox_data.append({
                "label": det.label,
                "area": w * h,
                "aspect_ratio": w / (h + 1e-6),
                "center_x": det.bounding_box[0] + (w/2),
                "center_y": det.bounding_box[1] + (h/2)
            })

df = pd.DataFrame(bbox_data)

# Plot 1: Area Distribution
plt.figure(figsize=(12, 5))
sns.histplot(df, x="area", hue="label", element="step", palette="magma", log_scale=True)
plt.title("Bounding Box Area Distribution (Log Scale)")
plt.xlabel("Relative Area (0 to 1)")
plt.show()

### 4. Spatial Heatmaps (Detection Centers)
Where are weapons usually located in your frames?

In [ ]:
plt.figure(figsize=(8, 8))
weapon_df = df[df['label'] == 'Weapon']
sns.kdeplot(data=weapon_df, x="center_x", y="center_y", cmap="Reds", fill=True, thresh=0.05)
plt.xlim(0, 1)
plt.ylim(1, 0) # Flip Y axis to match image coordinates
plt.title("Spatial Density of Weapon Detections")
plt.show()

### 5. Interactive Explorer (The FiftyOne App)
Use this to find the "Hardest" samples.

In [ ]:
# Launch App
session = fo.launch_app(dataset)

print("🚀 EDA v2 Dashboard Ready.")
print("Learning Tip: Filter for 'Confuser' labels. These are the objects that might trigger false alarms.")